# Chapter 13 — Skills

**Companion to *Pi Agents*** · [`Pi Agents` chapter 13](https://programmer.ie/books/pi/13-chapter/)

| | |
|---|---|
| Chapter | `13-chapter.md` · weight 13 · `/books/pi/13-chapter/` |
| Notebook | `notebooks/pi/13-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does the model's context carry a skill's **body**, or only enough to decide
whether it is relevant?**

This is the whole design of the feature, and it is checkable in one string: read
what the provider received and look for a marker planted in the body.


## What this notebook establishes

- The provider's system prompt carries a skill's **name**, **description** and **path**, and **not** its body. A marker string planted in the body never appears.
- A skill with **no description is not loaded at all** — a silent disappearance, not an error.
- `disable-model-invocation: true` hides a skill from the model while `/skill:name` still loads its body as the request.
- Skills are discovered **recursively**; two skills with the same name keep the first discovered, and only its description reaches the model.

## What this notebook does **not** establish

- **No routing quality is measured.** Pi exposes no way to score whether a description routed correctly, and `metadata/13-chapter.yaml` records that namespacing similar descriptions stays open (`ch13-q2`). Nothing here measures it.
- **Skills installed on the machine running the notebook are also discovered.** The chapter says so, and the driver therefore filters its report to the skills it created — otherwise a test asserting "exactly one skill" would fail on a developer machine and pass on CI.
- **A description is not a permission.** Nothing here establishes that the model will read a body it was offered.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real skills on disk, loaded by Pi's own resource loader, with the provider's system prompt read back.
* **Evidence scope:** `in_process_runtime`. Discovery and the description-in-context rule are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(13))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/13-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: one skill, three markers

The skill body contains `SECRET-BODY-TEXT`. The name, the description and the path
are planted separately so they can be looked for individually.

`systemPromptSeenByProvider` is the book's own helper: it runs one scripted turn and
returns the system message text exactly as the provider received it.

In [3]:
SKILLS = pinb.driver_source("ch13-skills.ts")
skills = pinb.driver(SKILLS, label="ch13-skills", timeout=420)
loaded = skills["loaded"]
print("skill:")
print("    name:        review-guard")
print("    description: Reviews changed code. Use when reviewing a diff.")
print("    body:        SECRET-BODY-TEXT")
print()
print(pinb.md_table(
    ["planted in the", "does the provider's system prompt contain it?"],
    [["name", "YES" if loaded["carriesName"] else "no"],
     ["description", "YES" if loaded["carriesDescription"] else "no"],
     ["path (SKILL.md)", "YES" if loaded["carriesPath"] else "no"],
     ["body", "YES" if loaded["carriesBody"] else "no"]],
))
print("discovered skills (this run's fixtures only):", ", ".join(loaded["discovered"]))

skill:
    name:        review-guard
    description: Reviews changed code. Use when reviewing a diff.
    body:        SECRET-BODY-TEXT

| planted in the | does the provider's system prompt contain it? |
|---|---|
| name | YES |
| description | YES |
| path (SKILL.md) | YES |
| body | no |
discovered skills (this run's fixtures only): review-guard


In [4]:
pinb.show_checks([
    pinb.check("the name reaches the model", loaded["carriesName"], "yes"),
    pinb.check("the description reaches the model", loaded["carriesDescription"], "yes"),
    pinb.check("the path reaches the model, so relative paths can be resolved",
               loaded["carriesPath"], "yes"),
    pinb.check("the body does not", not loaded["carriesBody"], "no: the marker is absent"),
    pinb.check("the skill was discovered", loaded["discovered"] == ["review-guard"],
               ", ".join(loaded["discovered"])),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------------  ------------------------
PASS  the name reaches the model  yes
PASS  the description reaches the model  yes
PASS  the path reaches the model, so relative paths can be resolved  yes
PASS  the body does not  no: the marker is absent
PASS  the skill was discovered  review-guard

5/5 assertions held.


That is the entire mechanism in one table. The model is told enough to decide
whether the skill is relevant, and nothing more. The body costs tokens only if
somebody reads it.

## Contrasting cases: three ways a skill stays out

Each row below is a separate session with a different fixture.

* **no description** — the skill is not loaded at all. There is no error and no
  warning; it is simply absent.
* **`disable-model-invocation: true`** — loaded, discoverable, and reachable by
  `/skill:name`, but absent from the model's context.
* **two skills with the same name** — one survives, and it is the first discovered.

In [5]:
noDesc = skills["noDescription"]
manual = skills["manualOnly"]
nested = skills["nested"]
collision = skills["collision"]

print(pinb.md_table(
    ["fixture", "discovered", "named in the model's context?"],
    [["no description", ", ".join(noDesc["discovered"]) or "(nothing)", "no" if not noDesc["mentionsName"] else "yes"],
     ["disable-model-invocation", ", ".join(manual["discovered"]) or "(nothing)", "no" if not manual["mentionsName"] else "yes"],
     ["nested two levels down", ", ".join(nested["discovered"]) or "(nothing)", "yes" if nested["mentionsName"] else "no"],
     ["two skills named review", ", ".join(collision["discovered"]) or "(nothing)",
      ("first" if collision["first"] else "second") if collision["first"] != collision["second"] else "both"]],
))

| fixture | discovered | named in the model's context? |
|---|---|---|
| no description | (nothing) | no |
| disable-model-invocation | manual-only | no |
| nested two levels down | nested-skill | yes |
| two skills named review | review | first |


In [6]:
pinb.show_checks([
    pinb.check("a skill with no description is not loaded at all",
               noDesc["discovered"] == [] and not noDesc["mentionsName"], "absent from both lists"),
    pinb.check("disable-model-invocation keeps it discoverable but invisible to the model",
               manual["discovered"] == ["manual-only"] and not manual["mentionsName"],
               "discovered, not described"),
    pinb.check("skills are discovered recursively",
               nested["discovered"] == ["nested-skill"] and nested["mentionsName"], "two levels down, found"),
    pinb.check("a name collision keeps exactly one skill, the first discovered",
               collision["discovered"] == ["review"] and collision["first"] != collision["second"],
               "first description present, second absent"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------------------------  ----------------------------------------
PASS  a skill with no description is not loaded at all  absent from both lists
PASS  disable-model-invocation keeps it discoverable but invisible to the model  discovered, not described
PASS  skills are discovered recursively  two levels down, found
PASS  a name collision keeps exactly one skill, the first discovered  first description present, second absent

4/4 assertions held.


## The other half: `/skill:name` loads the body on purpose

The body is not *forbidden* from the context — it is *not volunteered*. The
`<skill>` block below carries the body, the location, and the arguments the operator
typed after it.

In [7]:
sent = skills["explicitLoad"]
print("typed: /skill:manual-only src/auth only")
print()
print("the provider received:")
for line in sent.splitlines():
    print("   ", line)
print()

pinb.show_checks([
    pinb.check("the body is in the request", "MANUAL-BODY-TEXT" in sent, "present"),
    pinb.check("it arrives as a named block with a location",
               "<skill" in sent and "location=" in sent, "<skill name=... location=...>"),
    pinb.check("the operator's arguments are appended after the body",
               sent.rstrip().endswith("src/auth only"), "arguments last"),
    pinb.check("the block tells the model where it is", "References are relative to" in sent, "location stated"),
])

typed: /skill:manual-only src/auth only

the provider received:
    <skill name="manual-only" location="<tmp>">
    References are relative to <tmp>
    
    MANUAL-BODY-TEXT
    </skill>
    
    src/auth only

== Assertions ==
  assertion  observed
----  ----------------------------------------------------  -----------------------------
PASS  the body is in the request  present
PASS  it arrives as a named block with a location  <skill name=... location=...>
PASS  the operator's arguments are appended after the body  arguments last
PASS  the block tells the model where it is  location stated

4/4 assertions held.


## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [8]:
PATTERNS = 'ch13-skills/skills.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch13-skills/skills.test.ts', show="skill")

$ node --test ch13-skills/skills.test.ts
  PASS  6 passed, 0 failed, 0 skipped  in 989 ms

  tests matching 'skill':
    ok   the system prompt carries each skill's name, description and path, and not its body
    ok   a skill with no description is not loaded
    ok   /skill:name loads the body as the request, and arguments are appended
    ok   disable-model-invocation hides the skill from the model but not from /skill:name
    ok   skills are discovered recursively, unlike prompt templates
    ok   a name collision keeps the first discovered skill


## Your turn: predict, then run

**Predict first.** Move the body text into the **description**. Does the marker
reach the model now? What is the cost of doing that deliberately for a long skill?

**Then change one input.** `DESCRIPTION` below is what the skill advertises. Try
one that would plausibly route for a *different* task, and read the consequence.

**Predict the boundary.** A skill with no description is not loaded — silently. What
is the smallest change to the *loader* (not the skill) that would turn that
silence into a warning, and why did nobody add it?

In [9]:
DESCRIPTION = "Reviews changed code. Use when reviewing a diff."
BODY = "SECRET-BODY-TEXT"

print("advertised (always in context):", DESCRIPTION)
print("body (only when read):         ", BODY)
print()
print("rough size comparison, in characters rather than tokens:")
print(pinb.table(
    ["part", "chars", "share of the whole skill"],
    [["description", len(DESCRIPTION), f"{len(DESCRIPTION) / (len(DESCRIPTION) + len(BODY)) * 100:.0f}%"],
     ["body", len(BODY), f"{len(BODY) / (len(DESCRIPTION) + len(BODY)) * 100:.0f}%"]],
    indent="",
))

advertised (always in context): Reviews changed code. Use when reviewing a diff.
body (only when read):          SECRET-BODY-TEXT

rough size comparison, in characters rather than tokens:
part  chars  share of the whole skill
-----------  -----  ------------------------
description  48  75%
body  16  25%


## Interpretation

A skill is a **two-stage disclosure**:

```
always:    name + description + path          (cheap: a few dozen characters each)
on demand: the body, as a <skill> block      (expensive: whatever the author wrote)
```

The practical consequences:

1. **The description is the entire interface.** It is the only thing the model sees
   when deciding, so it is a routing decision written as prose. `ch13-q2` — how to
   keep two similar descriptions apart — is open, and nothing here measures routing
   quality.
2. **A missing description is not an error.** It is a skill that does not exist.
   When a skill "does not work", check the frontmatter first.
3. **Discovery depth differs from the previous mechanism.** Prompt templates are
   direct `.md` children; skills are recursive. If a skill "is not found", check the
   directory level before you check the name.

### What this chapter does not settle

Nothing here establishes that the model *reads* a body it was offered, or that it
reads the right one. That is a real-model question and this book has no
`real_model` evidence — the ledger's evidence scopes are `declarations`,
`in_process_runtime` and `shipped_binary`, and nothing else.

## Sources, execution mode and limitations

**Execution mode:** **Run** — six real skill fixtures discovered by Pi's own loader, with the provider's system prompt read back, plus the chapter's own test file.

**Evidence scope:** `in_process_runtime`.

### What was read or run

- **Ran** `drivers/ch13-skills.ts`: six sessions, each with its own temporary agent directory containing real `SKILL.md` files.
- **Read the provider's system prompt** via the book's `systemPromptSeenByProvider`, which captures it in a faux response factory.
- **Ran** `ch13-skills/skills.test.ts`.
- **Read** `examples/evidence.json`, chapter 13 rows (6 claims).

### Limitations

- **Skills installed on this machine are also discovered**, and the driver filters them out so the assertions are about the fixtures alone. This is the same trap the chapter's own tests guard against, and it is why a test of the form "exactly one skill was discovered" is unsafe on a developer machine.
- **No routing quality is measured.** Every result here is about *whether* a body is in context, never about whether the model would have chosen it.
- **The cost comparison in the exercise is in characters, not tokens**, and is reasoning material rather than a measurement.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
